# Laboratory – Logical Reasoning for Planning
## Using an LLM to Construct and Test a Simple Planning Agent

### Task 0: Understand the Planning Problem

**(a) What is the initial state I?**
$I = \{\text{At}(\text{Robot}, A), \text{At}(\text{Package}, A)\}$

**(b) What is the goal G?**
$G = \{\text{At}(\text{Package}, C)\}$

**(c) List the actions available to the robot.**
1. `Move(X, Y)` for connected locations.
2. `PickUp(Package, X)`
3. `Drop(Package, X)`

**(d) For each action, identify its preconditions and effects.**
- **Move(X, Y)**
  - Preconditions: `At(Robot, X)`, `Connected(X, Y)`
  - Effects: `¬At(Robot, X)`, `At(Robot, Y)`
- **PickUp(Package, X)**
  - Preconditions: `At(Robot, X)`, `At(Package, X)`
  - Effects: `¬At(Package, X)`, `Holding(Package)`
- **Drop(Package, X)**
  - Preconditions: `At(Robot, X)`, `Holding(Package)`
  - Effects: `¬Holding(Package)`, `At(Package, X)`

**Are the actions initially applicable?**
- `PickUp(Package, A)` is **applicable** because both preconditions `At(Robot, A)` and `At(Package, A)` are present in the initial state $I$.
- `Drop(Package, C)` is **not applicable** because neither of its preconditions (`At(Robot, C)` and `Holding(Package)`) are present in the initial state $I$.

---

### Task 1: Construct a Plan by Hand

We need to sequence actions to transport the package from A to C given connected locations $A \rightarrow B \rightarrow C$.

| State | Facts |
| :--- | :--- |
| $S_0$ | `At(Robot, A)`, `At(Package, A)` |
| **Action** | `PickUp(Package, A)` |
| $S_1$ | `At(Robot, A)`, `Holding(Package)` |
| **Action** | `Move(A, B)` |
| $S_2$ | `At(Robot, B)`, `Holding(Package)` |
| **Action** | `Move(B, C)` |
| $S_3$ | `At(Robot, C)`, `Holding(Package)` |
| **Action** | `Drop(Package, C)` |
| $S_4$ | `At(Robot, C)`, `At(Package, C)` |

---

### Task 2 & 3: LLM Planner Implementation and Testing
Below is the generated Python breadth-first search planner working on logical propositions.



In [ ]:
from collections import deque

class Action:
    def __init__(self, name, pre_pos, pre_neg, eff_pos, eff_neg):
        self.name = name
        self.pre_pos = set(pre_pos)
        self.pre_neg = set(pre_neg)
        self.eff_pos = set(eff_pos)
        self.eff_neg = set(eff_neg)

    def is_applicable(self, state):
        return self.pre_pos.issubset(state) and state.isdisjoint(self.pre_neg)

    def apply(self, state):
        return (state - self.eff_neg) | self.eff_pos

def bfs_plan(initial_state, goal_state, actions):
    # Queue stores: (current_state, plan_so_far, states_reached)
    queue = deque([(set(initial_state), [], [set(initial_state)])])
    visited = []

    while queue:
        current_state, plan, states = queue.popleft()

        # Goal check
        if set(goal_state).issubset(current_state):
            print("Plan found!")
            for i, (action, state) in enumerate(zip(plan, states[1:])):
                print(f"Step {i+1}: {action}")
                print(f"  State reached: {state}")
            return plan

        # Prevent cycles by checking if state was visited
        if current_state in visited:
            continue
        visited.append(current_state)

        for action in actions:
            if action.is_applicable(current_state):
                next_state = action.apply(current_state)
                queue.append((next_state, plan + [action.name], states + [next_state]))

    print("No plan found.")
    return None

# --- TESTS ---
def run_tests():
    print("=== Test A: Solvable Problem ===")
    actions = [
        Action("PickUp(Package, A)", ["At(Robot, A)", "At(Package, A)"], [], ["Holding(Package)"], ["At(Package, A)"]),
        Action("Move(A, B)", ["At(Robot, A)"], [], ["At(Robot, B)"], ["At(Robot, A)"]),
        Action("Move(B, C)", ["At(Robot, B)"], [], ["At(Robot, C)"], ["At(Robot, B)"]),
        Action("Drop(Package, C)", ["At(Robot, C)", "Holding(Package)"], [], ["At(Package, C)"], ["Holding(Package)"])
    ]
    initial = ["At(Robot, A)", "At(Package, A)"]
    goal = ["At(Package, C)"]
    bfs_plan(initial, goal, actions)

    print("\n=== Test B: Impossible Problem (No PickUp) ===")
    actions_impossible = [a for a in actions if not a.name.startswith("PickUp")]
    bfs_plan(initial, goal, actions_impossible)

    print("\n=== Test C: Irrelevant Actions ===")
    actions_irrelevant = actions + [
        Action("Move(B, A)", ["At(Robot, B)"], [], ["At(Robot, A)"], ["At(Robot, B)"])
    ]
    # The planner should not randomly just move back and forth or trick itself into thinking it's done
    bfs_plan(initial, goal, actions_irrelevant)

run_tests()



### Task 4: Logic and Search

**Explain where each is used:**
- **Logical reasoning:** Determines the constraints and rules of the universe. Specifically, it dictates `is_applicable` (checking if $S \models Preconditions(a)$ via subset logic) and state transitions (adding positive effects and removing negative effects).
- **Search:** Acts as the exploration engine. It iterates systematically (via Breadth-First Search) over all possible combinations of legally applicable actions branching out from the initial state until it discovers a sequence that satisfies the goal.

**Explanation of process:**
1. **Current state:** Check all available actions.
2. **Check action preconditions:** Use logic to test applicability.
3. **Apply effects:** Logic removes negative effects and adds positive effects.
4. **Generate successor state:** Represents the new state of the world.
5. **Search over alternatives:** Enqueue this new state in the BFS frontier to explore its future paths.
6. **Goal?:** Check if $S_n \models Goal$. If yes, terminate.

---

### Task 5: Can the LLM Verify Its Own Plan?

**(a) The LLM's explanation vs (b) Independently executed state transitions.**
You should absolutely trust **(b) the independently executed state transitions** more. 
LLMs function via probabilistic next-token prediction, which means they can hallucinate logical validity. They often confidently assert that a precondition is met even when it isn't, simply because it "looks" like a correct explanation structurally. A programmatic state-transition verification evaluates strict boolean logic unconditionally and deterministically, leaving zero room for hallucination.

---

### Task 6 & 8: Prolog and Logical Reasoning

**Task 6 Questions:**
- **(a) Why does Prolog return `true` for `can_move(a,b)`?** Because `connected(a,b)` is explicitly listed as a fact in the knowledge base, satisfying the rule `can_move(X,Y) :- connected(X,Y)`.
- **(b) Why does it not establish `can_move(a,c)`?** Because `connected(a,c)` is not in the knowledge base, and no combination of rules can logically deduce it. Thus, under the closed-world assumption, it evaluates to false.
- **(c) Relationship to logical implication:** `can_move(X,Y) :- connected(X,Y)` is the exact Prolog syntax equivalent of the first-order logic implication $\text{Connected}(X, Y) \implies \text{CanMove}(X, Y)$.

**Task 8 Questions:**
`wet_road` $\implies$ `slippery` $\implies$ `reduce_speed`. 
The query `?- reduce_speed.` succeeds because Prolog finds the rule `reduce_speed :- slippery`. It then tries to prove `slippery`. It finds the rule `slippery :- wet_road`. It then tries to prove `wet_road`, which exists purely as an absolute fact. The chain resolves to true.

---

### Reflection Questions

1. **Why is it useful to specify action preconditions and effects before asking an LLM to write the planner?** It grounds the prompt in formal logic, preventing the LLM from hallucinating "common sense" rules that break the deterministic nature of the planner.
2. **Give an example of an error that could occur if the planner failed to check an action's preconditions.** The robot could execute `Drop(Package, C)` without actually `Holding(Package)`, magically teleporting the package to C.
3. **Why is a plan that "looks reasonable" not necessarily a valid plan?** It might subtly violate a constraint, such as passing through a wall, moving twice in one step, or dropping an item it isn't holding.
4. **What did the LLM contribute to the implementation?** It successfully bridged the conceptual logic design with concrete Python data structures (sets, classes, deque) to build the BFS boilerplate and state transition functions.
5. **What did you have to verify independently?** That the planner correctly evaluates impossible plans as `No plan found` instead of crashing or hallucinating, and that it doesn't get stuck in infinite loops.
6. **In this laboratory, where is logical reasoning being used?** In the subset/disjoint checks for precondition validation, and the set union/difference operations for state effects.
7. **How is planning related to the search algorithms studied in the previous module?** Planning is essentially pathfinding (search) through an abstract graph where nodes are "world states" and edges are "actions", exactly analogous to pathfinding through a physical grid where nodes are "locations" and edges are "movements".

### 7.2 Prolog Reflection
1. **Difference between fact and rule?** A fact is unconditionally true (e.g. `wet_road.`). A rule establishes a conditional implication (e.g. `slippery :- wet_road.`).
2. **How does a query correspond to asking if something follows from a KB?** A query asks the inference engine to attempt to prove a statement by tracing it back to known facts via logical rules.
3. **Why verify a Python plan with Prolog?** Because the Python generator might have bugs in its state-management logic. Prolog provides a declarative, mathematically sound inference engine to independently prove the plan is sound.
4. **Advantage of an independent verifier?** It isolates the "generator" (which can hallucinate or have search bugs) from the "evaluator" (which strictly enforces logical rules).

